In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Notebook 01: SAMSum Dataset Exploration
Objective:
- Download SAMSum dataset
- Inspect structure and samples
- Understand dialogue–summary pairs
- Prepare for data ingestion pipeline

In [3]:
!pip install -r /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=2fb4d722c5c2f749108181f01f706ecb380ba22c8a5b288e1a3e11cc37c5f53e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [4]:
import sys
print("Python version:", sys.version)

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [5]:
from datasets import load_dataset
import pandas as pd

In [7]:
dataset = load_dataset("knkarthick/samsum")
dataset

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [10]:
for split in dataset:
    print(f"{split} size: {len(dataset[split])}")

train size: 14731
validation size: 818
test size: 819


In [11]:
dataset["train"][0]

{'id': '13818513',
 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}

In [12]:
train_df = pd.DataFrame(dataset["train"])
train_df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."


In [13]:
print("Dialogue:\n")
print(train_df.loc[0, "dialogue"])

print("\nSummary:\n")
print(train_df.loc[0, "summary"])

Dialogue:

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Summary:

Amanda baked cookies and will bring Jerry some tomorrow.


In [14]:
train_df["dialogue_word_count"] = train_df["dialogue"].apply(lambda x: len(x.split()))
train_df["summary_word_count"] = train_df["summary"].apply(lambda x: len(x.split()))

train_df[["dialogue_word_count", "summary_word_count"]].describe()

,dialogue_word_count,summary_word_count
count,14731.000000,14731.000000
mean,93.792750,20.318444
std,74.031937,11.153570
min,7.000000,1.000000
25%,39.000000,12.000000
50%,73.000000,18.000000
75%,128.000000,27.000000
max,803.000000,64.000000


### Observations
- Dialogues are informal and multi-turn
- Summaries are concise and abstractive
- Large variance in dialogue length
- Confirms need for transformer-based summarization

In [15]:
from pathlib import Path

RAW_DATA_DIR = Path("/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_DIR

PosixPath('/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw')

In [16]:
for split in dataset.keys():
    output_path = RAW_DATA_DIR / f"samsum_{split}.json"
    dataset[split].to_json(output_path)
    print(f"Saved {split} split to {output_path}")

Creating json from Arrow format:   0%|          | 0/15 [00:00<?, ?ba/s]

Saved train split to /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw/samsum_train.json


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved validation split to /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw/samsum_validation.json


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved test split to /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw/samsum_test.json


- Files in `artifacts/data/raw/` are immutable
- No preprocessing is applied at this stage
- All future pipelines must read from these files
- Dataset source: Hugging Face (knkarthick/samsum)